# Dotlas US restaurants (Databricks Marketplace)

**Setup:** `pip install -r requirements.txt` from the **Nuri_App** folder.

**Credentials:** Put `DATABRICKS_HOST`, `DATABRICKS_HTTP_PATH`, and `DATABRICKS_TOKEN` in `.env` at the repo root (the next cell loads them). Or export them in the shell / Jupyter kernel env.

**Token (fixes `required scopes: sql`):** In **Settings → Developer → Access tokens → Generate new token**, either leave **API scopes empty** (token gets full workspace permissions, including SQL), **or** add the **`sql`** scope explicitly. A *scoped* token without `sql` will fail for this notebook. After changing `.env`, re-run the **first** code cell (`load_dotenv` uses `override=True` so the new token loads).

**Kernel working directory:** Open/run Jupyter with working directory **Nuri_App** (folder with `package.json`) so `.env` and `src/lib/databricks_client.py` are found.

**`TABLE_OR_VIEW_NOT_FOUND`:** The default path is not in your metastore until the Marketplace listing is installed and you use the real name. **Important:** In `.env`, `DOTLAS_RESTAURANTS_TABLE` must be on a line that does **not** start with `#` — commented lines are ignored by `load_dotenv` (a common mistake). (1) Install Dotlas from [Marketplace](https://marketplace.databricks.com/). (2) In **Catalog**, copy the full `catalog.schema.table` for `us_restaurants`. (3) Set it in `.env` as `DOTLAS_RESTAURANTS_TABLE=...` **or** paste it into `_DOTLAS_FROM_NOTEBOOK` in the setup cell below. (4) Re-run setup, then query. The **search** cell lists matching table names if `information_schema` works.

In [19]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "lib" / "databricks_client.py").exists():
        return cwd
    if (cwd.parent / "src" / "lib" / "databricks_client.py").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not find src/lib/databricks_client.py. "
        "Open/run Jupyter with working directory set to Nuri_App (repo root)."
    )


ROOT = _repo_root()
# override=True: updating .env and re-running this cell replaces env vars (default dotenv does not).
load_dotenv(ROOT / ".env", override=True)

# Paste your Unity Catalog 3-part name here if .env is wrong or commented (e.g. "main.dotlas_samples.us_restaurants").
# Takes precedence over .env when non-empty.
_DOTLAS_FROM_NOTEBOOK = ""
if _DOTLAS_FROM_NOTEBOOK.strip():
    os.environ["DOTLAS_RESTAURANTS_TABLE"] = _DOTLAS_FROM_NOTEBOOK.strip()

_lib = ROOT / "src" / "lib"
if str(_lib) not in sys.path:
    sys.path.insert(0, str(_lib))

import importlib

import databricks_client
importlib.reload(databricks_client)
from databricks_client import DatabricksClient

_dotlas = (os.environ.get("DOTLAS_RESTAURANTS_TABLE") or "").strip()
print(
    "DOTLAS_RESTAURANTS_TABLE:",
    _dotlas or "(not set — using default; add to .env if TABLE_OR_VIEW_NOT_FOUND)",
)

# Optional in .env: DOTLAS_RESTAURANTS_TABLE=catalog.schema.us_restaurants (required if default table missing)
# When CITY is set: ROWS=None uses env DOTLAS_DEFAULT_CITY_LIMIT (default 25000); or set ROWS explicitly.
# When CITY is None: set ROWS to a number (random US sample).
ROWS = 10_000
CITY = "San Francisco"  # set to None for a random US sample (no city filter)

DOTLAS_RESTAURANTS_TABLE: (not set — using default; add to .env if TABLE_OR_VIEW_NOT_FOUND)


In [20]:
# Optional: find the real 3-part table name (run after setup cell). Then set DOTLAS_RESTAURANTS_TABLE in .env.
host = os.environ.get("DATABRICKS_HOST")
http_path = os.environ.get("DATABRICKS_HTTP_PATH")
token = os.environ.get("DATABRICKS_TOKEN")
if host and http_path and token:
    _probe = DatabricksClient(host, http_path, token)
    try:
        _hits = _probe.find_tables_in_information_schema("us_restaurant", limit=100)
        print("Tables matching 'us_restaurant' (copy catalog.schema.table into DOTLAS_RESTAURANTS_TABLE):")
        display(_hits if not _hits.empty else "(none — install Dotlas from Marketplace or browse Catalog)")
    except Exception as e:
        print("information_schema search failed (use Catalog Explorer instead):", e)
else:
    print("Set DATABRICKS_* env vars in the setup cell first.")

Tables matching 'us_restaurant' (copy catalog.schema.table into DOTLAS_RESTAURANTS_TABLE):


,table_catalog,table_schema,table_name
0,dotlas_restaurants_in_the_united_states_of_ame...,samples,us_restaurants


In [21]:
host = os.environ.get("DATABRICKS_HOST")
http_path = os.environ.get("DATABRICKS_HTTP_PATH")
token = os.environ.get("DATABRICKS_TOKEN")
if not host or not http_path or not token:
    raise RuntimeError(
        "Set DATABRICKS_HOST, DATABRICKS_HTTP_PATH, and DATABRICKS_TOKEN in your environment."
    )

client = DatabricksClient(host, http_path, token)
print("Querying table:", client.dotlas_restaurants_table)
df = client.get_dotlas_restaurants(limit=ROWS, city=CITY)
df.shape

Querying table: dotlas_restaurants_in_the_united_states.samples.us_restaurants


ServerOperationError: [TABLE_OR_VIEW_NOT_FOUND] The table or view `dotlas_restaurants_in_the_united_states`.`samples`.`us_restaurants` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 14

In [ ]:
df.head(10)

In [ ]:
df.info()

## Exploration

Add cells below for filters, `value_counts`, plots, exports, etc.

In [ ]:
# Example: with CITY="San Francisco" above, `df` is already SF-only.
# If you used CITY=None, filter with: df[df["city"].str.lower() == "san francisco"]
df.head(50)